# `indic/02` — Apply IndicXlit Romanisation

**Purpose:** Apply the IndicXlit neural transliteration system (Madhani et al., 2023)
to the target side of the IndicMT Eval corpus, converting native-script `hyp` and
`ref` columns into Latin script.

The intervention is **script-only**: all phonological, morphological, and semantic
content is preserved. Source sentences (`src`) and MQM scores are left untouched,
so any downstream change in a metric score is attributable exclusively to
orthographic form.

**Reads:** `data/indic/<ISO>_indicmt.csv` (produced by `indic/01`)

**Outputs produced:**
```
data/indic/GUJ_indicmt.csv   -- adds columns: hyp_rom, ref_rom
data/indic/HIN_indicmt.csv   -- adds columns: hyp_rom, ref_rom
data/indic/MAL_indicmt.csv   -- adds columns: hyp_rom, ref_rom
data/indic/MAR_indicmt.csv   -- adds columns: hyp_rom, ref_rom
data/indic/TAM_indicmt.csv   -- adds columns: hyp_rom, ref_rom
```

**IndicXlit language codes used:**

| ISO | Language | Script | IndicXlit code |
|-----|----------|--------|----------------|
| GUJ | Gujarati | Gujarati | `gu` |
| HIN | Hindi | Devanagari | `hi` |
| MAL | Malayalam | Malayalam | `ml` |
| MAR | Marathi | Devanagari | `mr` |
| TAM | Tamil | Tamil | `ta` |

In [ ]:
import subprocess, sys

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["ai4bharat-transliteration", "pandas"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg}..."); _install(pkg)

print("Dependencies ready.")

## Configuration

All paths are relative to the repository root.  
The notebook reads the CSVs written by `indic/01` and overwrites them
in place, adding `hyp_rom` and `ref_rom` columns.

In [ ]:
from pathlib import Path
import pandas as pd

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("data/indic")

# ── Language config: ISO code -> IndicXlit lang code ──────────────────────
LANG_CONFIGS = {
    "GUJ": "gu",
    "HIN": "hi",
    "MAL": "ml",
    "MAR": "mr",
    "TAM": "ta",
}

# ── Columns to romanise ───────────────────────────────────────────────────
# hyp = MT hypothesis, ref = human reference
# src (English) is intentionally excluded
TARGET_COLS = {"hyp": "hyp_rom", "ref": "ref_rom"}

print(f"Data directory : {DATA_DIR.resolve()}")
print(f"Languages      : {list(LANG_CONFIGS.keys())}")
print(f"Columns        : {list(TARGET_COLS.keys())} -> {list(TARGET_COLS.values())}")

## Step 1 — Load Per-Language CSVs

Read the five files produced by `indic/01` and confirm they contain
the expected `hyp` and `ref` columns before starting transliteration.

In [ ]:
raw = {}  # iso -> pd.DataFrame

for iso in LANG_CONFIGS:
    path = DATA_DIR / f"{iso}_indicmt.csv"
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run indic/01_fetch_indicmt_eval.ipynb first.")
    df = pd.read_csv(path)
    raw[iso] = df
    present  = [c for c in TARGET_COLS if c in df.columns]
    missing  = [c for c in TARGET_COLS if c not in df.columns]
    print(f"  {iso}  {len(df):,} rows  |  found: {present}  |  missing: {missing}")

print(f"\nLoaded {len(raw)} language files.")

## Step 2 — Initialise IndicXlit Engines

One `XlitEngine` instance is created per language and reused across all rows.
Loading is done up-front so any missing model weights are caught before
transliteration begins.

IndicXlit internally uses a sequence-to-sequence model trained on the
Aksharantar dataset (Madhani et al., 2023). We use `beam_width=4` and
`rescore=True` to obtain the single best transliteration per token.

In [ ]:
from ai4bharat.transliteration import XlitEngine

engines = {}  # iso -> XlitEngine

for iso, lang_code in LANG_CONFIGS.items():
    print(f"  Loading engine for {iso} ({lang_code})...", end=" ", flush=True)
    engines[iso] = XlitEngine(lang_code, beam_width=4, rescore=True)
    print("ready.")

print("\nAll engines loaded.")

## Step 3 — Transliterate `hyp` and `ref`

For each language we apply `engine.translit_sentence()` row-by-row to both
the MT hypothesis and the human reference. The function handles word-level
segmentation internally.

Empty or NaN cells are passed through as empty strings. If a row raises
an exception (e.g., a sentence that is already in Latin script), the
original text is kept and a warning is printed so execution continues.

In [ ]:
import warnings


def translit_series(series, engine, lang_code):
    """Transliterate a pandas Series using IndicXlit; return a new Series."""
    out = []
    for idx, text in series.items():
        if pd.isna(text) or str(text).strip() == "":
            out.append("")
            continue
        try:
            out.append(engine.translit_sentence(str(text), lang_code))
        except Exception as exc:
            warnings.warn(f"Row {idx}: {exc} -- keeping original text")
            out.append(str(text))
    return pd.Series(out, index=series.index)


romanised = {}  # iso -> pd.DataFrame with hyp_rom / ref_rom added

for iso, df in raw.items():
    lang_code = LANG_CONFIGS[iso]
    engine    = engines[iso]
    d         = df.copy()

    print(f"{'='*55}")
    print(f"{iso} ({lang_code})  --  {len(d):,} rows")
    print(f"{'='*55}")

    for src_col, out_col in TARGET_COLS.items():
        if src_col not in d.columns:
            print(f"  '{src_col}' not found -- skipped")
            continue
        if out_col in d.columns:
            print(f"  '{out_col}' already exists -- skipped (re-run to overwrite)")
            continue

        print(f"  Romanising '{src_col}' -> '{out_col}'...", end=" ", flush=True)
        rom = translit_series(d[src_col], engine, lang_code)
        filled = rom.str.strip().str.len().gt(0).sum()
        print(f"done  ({filled:,}/{len(d):,} non-empty)")

        insert_pos = d.columns.get_loc(src_col) + 1
        d.insert(insert_pos, out_col, rom)

    romanised[iso] = d
    print()

## Step 4 — Sanity Check

Print three random rows per language showing native `hyp` alongside romanised
`hyp_rom`. This is a manual phonological consistency check; IndicXlit output
is deterministic for a given model checkpoint so no assertion is needed here.

In [ ]:
N_SAMPLES = 3

for iso, df in romanised.items():
    if "hyp" not in df.columns or "hyp_rom" not in df.columns:
        continue
    sample = df[["hyp", "hyp_rom"]].dropna().sample(
        min(N_SAMPLES, len(df)), random_state=42
    )
    print(f"{'='*55}")
    print(f"{iso} -- hyp (native) vs hyp_rom (romanised)")
    print(f"{'='*55}")
    for _, row in sample.iterrows():
        print(f"  native : {row['hyp']}")
        print(f"  roman  : {row['hyp_rom']}")
        print()

## Step 5 — Save Updated CSVs

The romanised columns are written back into the same per-language CSV files
in `data/indic/`. All downstream notebooks (`03` onwards) will find
`hyp_rom` and `ref_rom` alongside the original columns.

In [ ]:
for iso, df in romanised.items():
    out_path = DATA_DIR / f"{iso}_indicmt.csv"
    df.to_csv(out_path, index=False)
    new_cols = [c for c in df.columns if c.endswith("_rom")]
    print(f"  Saved  {out_path}  ({len(df):,} rows)  new cols: {new_cols}")

print(f"\n  {len(romanised)} files updated in {DATA_DIR.resolve()}")

## Step 6 — Romanisation Coverage Summary

Report fill rates for `hyp_rom` and `ref_rom` across all five languages.

In [ ]:
print(f"{'Lang':6s}  {'Rows':>6s}  {'hyp_rom filled':>16s}  {'ref_rom filled':>16s}")
print("-" * 52)

for iso, df in romanised.items():
    n = len(df)
    hyp_filled = (df["hyp_rom"].str.strip().str.len().gt(0).sum()
                  if "hyp_rom" in df.columns else 0)
    ref_filled = (df["ref_rom"].str.strip().str.len().gt(0).sum()
                  if "ref_rom" in df.columns else 0)
    print(f"  {iso:4s}    {n:>5,}    {hyp_filled:>6,} / {n:<5,}    {ref_filled:>6,} / {n:<5,}")

print("\n  Romanisation complete. data/indic/ is ready for indic/03.")